# Fase 3: Rotulação ótima das amostras com o oráculo

Nesta fase atribuimos a cada estado do tabuleiro o seu gabarito de jogo perfeito. Diferente das fases anteriores, aqui não usamos aproximada com profundidade
limitada: usamos um oráculo que resolve o jogo dos pontinhos no tabuleiro pequeno (4 por 3) de forma completa e exata.

O oráculo é uma tabela de base de dados pré-calculada por análise retrógrada. Para cada uma das 2^31 configurações possíveis de traços ela guarda o valor exato da
posição sob jogo ótimo dos dois lados, ou seja, a diferença futura de caixas a favor de quem vai jogar, contando apenas as caixas que ainda serão fechadas dali
até o fim. Com essa tabela pronta, avaliar qualquer posição vira uma simples consulta em vetor, sem nenhuma busca.

A base completa ocupa cerca de 2 GB, o que é inviável de anexar. Por isso, junto desta entrega vai uma amostra do oráculo: um arquivo pequeno com os valores
exatos apenas dos estados necessários para rotular o NPZ de exemplo da pasta de dados. O notebook detecta sozinho se carregou a tabela completa ou a amostra.

Para cada amostra eu preencho três campos:

- `score_melhor_jogada`: vetor com 31 posições com o valor exato de cada traço ainda disponível. Traços já ocupados recebem o sentinela -1e9.
- `melhor_jogada`: o rótulo do traço escolhido. Quando há mais de um traço com o mesmo valor máximo, sorteio um deles de forma uniforme, já que todos são igualmente ótimos. O sorteio usa semente fixa, então o resultado é reproduzível.
- `depth_melhor_jogada`: a distância até o fim do jogo, isto é, quantos lances ainda faltam para o tabuleiro ficar completo. Como o oráculo enxerga a partida inteira, esse valor é exatamente `31 - qtd_tracos`. Ele indica até onde o score da melhor jogada consegue enxergar o futuro: como o jogo é resolvido por completo, a visão alcança sempre o término da partida.

Os demais campos das amostras (estados, canais, parâmetros de cadeias e os scores de geração) são preservados sem alteração. A gravação é feita no próprio arquivo.

## Por que o oráculo é exato

No tabuleiro pequeno existem 31 traços, logo no máximo 2^31 estados distintos. Esse número é grande mas tratável: percorrendo os estados do mais cheio para o mais
vazio, o valor de cada posição depende apenas de posições com um traço a mais, que já foram resolvidas. Essa é a ideia da análise retrógrada.

A convenção de valor segue a regra do jogo: um lance que fecha pelo menos uma caixa mantém a vez de quem jogou, então o valor é a quantidade de caixas fechadas mais o
valor da posição filha. Um lance que não fecha caixa passa a vez, e o valor é o negativo do valor da filha. A posição final, com o tabuleiro cheio, vale zero.

In [1]:
import os
import time
from pathlib import Path

import numpy as np

# Tabuleiro pequeno: 4 linhas por 3 colunas de caixas. Isso vira uma matriz 9 por 7
# e dá 31 traços no total.
LINHAS, COLUNAS = 4, 3
N_TRACOS = 31
ALTURA = 2 * LINHAS + 1     # 9
LARGURA = 2 * COLUNAS + 1   # 7

# Pasta com os NPZ a rotular. A regravação acontece no próprio arquivo.
PASTA_DADOS = Path.cwd() / "dados" / "base_limpa"
PADRAO_ARQUIVOS = "dataset_pequeno_*.npz"

# Arquivo do oráculo. Pode ser a tablebase completa (vetor com 2^31 posições) ou a
# amostra entregue junto, que contém apenas os estados necessários para rotular os
# NPZ desta pasta. O código detecta sozinho qual dos dois foi carregado.
ORACULO_PATH = PASTA_DADOS / "tablebase_amostra_pequeno_4x3.npy"

# Semente do sorteio que desempata lances igualmente ótimos. Somo o índice do
# arquivo para variar entre arquivos sem perder a reprodutibilidade.
SEMENTE_BASE = 20260531

SENTINELA = np.float32(-1e9)

print("Tabuleiro    :", f"{LINHAS} x {COLUNAS}  ({N_TRACOS} traços, matriz {ALTURA} x {LARGURA})")
print("Oráculo      :", ORACULO_PATH)
print("Pasta dados  :", PASTA_DADOS)
print("Padrão       :", PADRAO_ARQUIVOS)

Tabuleiro    : 4 x 3  (31 traços, matriz 9 x 7)
Oráculo      : D:\Pos_Graduacao\Trabalho_Final\gerador_dados\dados\base_limpa\tablebase_amostra_pequeno_4x3.npy
Pasta dados  : D:\Pos_Graduacao\Trabalho_Final\gerador_dados\dados\base_limpa
Padrão       : dataset_pequeno_*.npz


In [2]:
ARANGE = np.arange(N_TRACOS)
DUAS_COLUNAS = np.arange(2)[None, None, :]


# Mapeamento de cada traço para as caixas que ele toca.
# Cada caixa tem o centro numa posição (linha ímpar, coluna ímpar) e é fechada por
# quatro traços: o de cima, o de baixo, o da esquerda e o da direita. Represento o
# conjunto dos quatro traços de uma caixa como uma máscara de bits sobre os 31 bits.
# Cada traço toca uma ou duas caixas.

def construir_mapeamento(labels):
    indice = {lbl: i for i, lbl in enumerate(labels)}
    n = len(labels)
    edge_rc = np.array([(int(p[1]), int(p[2])) for p in (l.split("_") for l in labels)])

    caixas_por_traco = {i: [] for i in range(n)}
    for br in range(1, ALTURA, 2):
        for bc in range(1, LARGURA, 2):
            tracos_da_caixa = [
                f"H_{br - 1}_{bc}",   # cima
                f"H_{br + 1}_{bc}",   # baixo
                f"V_{br}_{bc - 1}",   # esquerda
                f"V_{br}_{bc + 1}",   # direita
            ]
            bits = [indice[t] for t in tracos_da_caixa]
            mascara = 0
            for b in bits:
                mascara |= (1 << b)
            for b in bits:
                caixas_por_traco[b].append(mascara)

    mascaras = np.zeros((n, 2), dtype=np.int64)
    qtd_caixas = np.zeros(n, dtype=np.int64)
    for e in range(n):
        ms = caixas_por_traco[e]
        qtd_caixas[e] = len(ms)
        for k, m in enumerate(ms):
            mascaras[e, k] = m
    return edge_rc, mascaras, qtd_caixas


def bitmasks_do_lote(estados, edge_rc):
    # Converte cada matriz num inteiro de 31 bits: o bit i fica ligado quando o
    # traço i (na ordem canônica) já foi colocado, isto é, a célula não está vazia.
    celulas = estados[:, edge_rc[:, 0], edge_rc[:, 1]]   # (N, 31)
    return ((celulas != 0).astype(np.int64) << ARANGE).sum(1)


# Carrega o oráculo e monta a função de consulta. A tablebase completa é um vetor
# indexado direto pelo bitmask; a amostra é um par (bitmasks ordenados, valores),
# consultado por busca binária.
_oraculo = np.load(str(ORACULO_PATH))
if _oraculo.ndim == 1:
    consultar_oraculo = lambda bits: _oraculo[bits]
    print(f"Tablebase completa carregada: {_oraculo.shape[0]:,} posições.")
else:
    _chaves, _valores = _oraculo[0], _oraculo[1]
    consultar_oraculo = lambda bits: _valores[np.searchsorted(_chaves, bits)]
    print(f"Amostra do oráculo carregada: {_chaves.shape[0]:,} estados.")


def scores_exatos_do_lote(estado_bits, mascaras, qtd_caixas):
    # Para cada estado e cada traço, calculo o valor exato do lance consultando o
    # oráculo na posição filha (estado com aquele traço a mais).
    # - lance que fecha caixa mantém a vez: valor = caixas fechadas + valor da filha
    # - lance que não fecha caixa passa a vez: valor = - valor da filha
    # Traços já ocupados recebem o sentinela.
    filhos = estado_bits[:, None] | (np.int64(1) << ARANGE)[None, :]   # (N, 31)
    valor_filho = consultar_oraculo(filhos).astype(np.int64)

    fecha = (((filhos[:, :, None] & mascaras[None, :, :]) == mascaras[None, :, :])
             & (DUAS_COLUNAS < qtd_caixas[None, :, None])).sum(2)             # (N, 31)
    valor = np.where(fecha > 0, fecha + valor_filho, -valor_filho)

    ocupado = ((estado_bits[:, None] >> ARANGE) & 1) == 1
    return np.where(ocupado, SENTINELA, valor).astype(np.float32)

Amostra do oráculo carregada: 261,148 estados.


In [3]:
# Lista os arquivos a rotular. Excluo as variantes de simetria (geradas na fase 4)
# e eventuais arquivos temporários de uma execução interrompida.
SUFIXOS_SIMETRIA = ("_refH", "_refV", "_r180")
arquivos = sorted(PASTA_DADOS.glob(PADRAO_ARQUIVOS))
arquivos = [
    p for p in arquivos
    if not any(p.stem.endswith(s) for s in SUFIXOS_SIMETRIA)
    and ".tmp." not in p.name
]

print(f"{len(arquivos)} arquivo(s) a rotular em {PASTA_DADOS}")
for p in arquivos[:3]:
    print("  ", p.name)
if len(arquivos) > 3:
    print("   ...")
    print("  ", arquivos[-1].name)

1 arquivo(s) a rotular em D:\Pos_Graduacao\Trabalho_Final\gerador_dados\dados\base_limpa
   dataset_pequeno_0001.npz


In [4]:
# Loop principal. Para cada arquivo eu recalculo os três campos de rótulo a partir
# do oráculo e regravo, preservando todo o resto do schema. A gravação é atômica:
# escrevo um arquivo .tmp e só então substituo o original, de modo que uma
# interrupção no meio não deixa o NPZ corrompido.
labels_canonicos = None
edge_rc = mascaras = qtd_caixas = LAB = None
total_estados = 0
t_total = time.time()

for n_arq, path in enumerate(arquivos):
    with np.load(path, allow_pickle=False) as d:
        dados = dict(d)

    # O mapeamento e os labels são os mesmos para todos os arquivos; calculo uma vez.
    if labels_canonicos is None:
        labels_canonicos = list(dados["labels_canonicos"])
        LAB = np.array(labels_canonicos, dtype="<U5")
        edge_rc, mascaras, qtd_caixas = construir_mapeamento(labels_canonicos)

    estados = dados["estados"]
    qtd_tracos = dados["qtd_tracos"].astype(np.int64)
    N = len(estados)
    total_estados += N

    estado_bits = bitmasks_do_lote(estados, edge_rc)
    score_mj = scores_exatos_do_lote(estado_bits, mascaras, qtd_caixas)

    # melhor_jogada: sorteio uniforme entre os lances de valor máximo. O ruído só é
    # positivo nas posições ótimas, então o argmax cai sempre num lance ótimo.
    rng = np.random.default_rng(SEMENTE_BASE + n_arq)
    mascarado = np.where(score_mj > -1e8, score_mj, -np.inf)
    eh_otimo = mascarado == mascarado.max(axis=1, keepdims=True)
    ruido = np.where(eh_otimo, rng.random(score_mj.shape), -1.0)
    melhor_jogada = LAB[ruido.argmax(axis=1)]

    # depth_melhor_jogada: distância até o fim do jogo (lances restantes).
    depth_mj = (N_TRACOS - qtd_tracos).astype(np.int8)

    dados["melhor_jogada"] = melhor_jogada
    dados["score_melhor_jogada"] = score_mj
    dados["depth_melhor_jogada"] = depth_mj

    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **dados)
    os.replace(tmp, path)
    print(f"[{n_arq + 1:3d}/{len(arquivos)}] {path.name}: {N:,} estados rotulados.")

print(f"\nConcluído: {total_estados:,} estados em {len(arquivos)} arquivos, {(time.time() - t_total) / 60:.1f} min.")

[  1/1] dataset_pequeno_0001.npz: 20,000 estados rotulados.

Concluído: 20,000 estados em 1 arquivos, 0.0 min.


In [5]:
# Auditoria final. Releio os arquivos gravados e confiro três invariantes:
#   1. nenhuma melhor_jogada vazia;
#   2. depth_melhor_jogada igual a 31 - qtd_tracos em todo estado;
#   3. a melhor_jogada escolhida é sempre um lance de valor máximo.
indice = {l: i for i, l in enumerate(labels_canonicos)}
problemas = []
estados_auditados = 0

for path in arquivos:
    with np.load(path, allow_pickle=False) as d:
        mj = d["melhor_jogada"]
        sc = d["score_melhor_jogada"]
        depth = d["depth_melhor_jogada"].astype(np.int64)
        qt = d["qtd_tracos"].astype(np.int64)
    N = len(mj)
    estados_auditados += N

    if (mj == "").any():
        problemas.append(f"{path.name}: existe melhor_jogada vazia")
    if not np.array_equal(depth, N_TRACOS - qt):
        problemas.append(f"{path.name}: depth_melhor_jogada diferente de 31 - qtd_tracos")

    mji = np.array([indice[m] for m in mj])
    mascarado = np.where(sc > -1e8, sc, -np.inf)
    escolhido = mascarado[np.arange(N), mji]
    if not np.all(escolhido == mascarado.max(axis=1)):
        problemas.append(f"{path.name}: melhor_jogada não ótima em algum estado")

if problemas:
    print(f"FALHA: {len(problemas)} problema(s):")
    for p in problemas[:20]:
        print("  ", p)
else:
    print(f"OK: {len(arquivos)} arquivos auditados, {estados_auditados:,} estados.")
    print("Todos os rótulos são ótimos, depth = 31 - qtd_tracos e nenhuma melhor_jogada vazia.")
    print("Pronto para a fase 4 (augmentação por simetria).")

OK: 1 arquivos auditados, 20,000 estados.
Todos os rótulos são ótimos, depth = 31 - qtd_tracos e nenhuma melhor_jogada vazia.
Pronto para a fase 4 (augmentação por simetria).
